In [1]:
import os
import shutil
from pdf2image import pdfinfo_from_path, convert_from_path
from ocrmac import ocrmac

# Define main folder paths
INPUT_FOLDER = "./CBB_Documents"  # Root folder containing subfolders (2004, 2005, etc.)
OUTPUT_ROOT = "./ocr_results"               # Text files will mirror subfolders here
DONE_ROOT = "./processed_pdfs"              # Moved PDFs will mirror subfolders here

# Collect all PDF files across all subfolders first to get an accurate count
pdf_tasks = []
for root, dirs, files in os.walk(INPUT_FOLDER):
    for file in files:
        if file.lower().endswith(".pdf"):
            full_path = os.path.join(root, file)
            # Calculate the relative path from the input folder (e.g., "2004/doc.pdf")
            rel_path = os.path.relpath(full_path, INPUT_FOLDER)
            pdf_tasks.append((full_path, rel_path))

if not pdf_tasks:
    print(f"No PDF files found inside '{INPUT_FOLDER}' or its subfolders.")
    exit()

print(f"Found {len(pdf_tasks)} PDF(s) across all folders to process.\n" + "="*40)

# Loop through each PDF file task
for file_index, (pdf_path, rel_path) in enumerate(pdf_tasks, 1):
    file_name = os.path.basename(pdf_path)
    
    # Mirror the subfolder structure for outputs
    rel_dir = os.path.dirname(rel_path)
    current_output_dir = os.path.join(OUTPUT_ROOT, rel_dir)
    current_done_dir = os.path.join(DONE_ROOT, rel_dir)
    
    # Ensure destination subfolders exist
    os.makedirs(current_output_dir, exist_ok=True)
    os.makedirs(current_done_dir, exist_ok=True)
    
    # Define exact output targets
    txt_name = os.path.splitext(file_name)[0] + ".txt"
    output_txt_path = os.path.join(current_output_dir, txt_name)
    destination_pdf_path = os.path.join(current_done_dir, file_name)
         
    print(f"\n[{file_index}/{len(pdf_tasks)}] Processing: {rel_path}")
         
    try:
        # Get total page count
        info = pdfinfo_from_path(pdf_path)
        total_pages = info["Pages"]
        print(f"-> Total pages: {total_pages}")
    except Exception as e:
        print(f"-> Error reading PDF info for {file_name}: {e}. Skipping file.")
        continue

    # Open the specific output file for this PDF
    with open(output_txt_path, "w", encoding="utf-8") as f:
        # Process page by page
        for i in range(1, total_pages + 1):
            print(f"   Processing page {i}/{total_pages}...")
                         
            # Convert ONLY the current page into memory
            try:
                single_page_list = convert_from_path(
                    pdf_path, 
                    dpi=300, 
                    first_page=i, 
                    last_page=i
                )
            except Exception as e:
                print(f"   Error converting page {i}: {e}")
                continue
                             
            if not single_page_list:
                continue
                             
            page_image = single_page_list[0]
            temp_img_path = f"temp_page_{file_index}_{i}.png"
                         
            # Save single page to disk
            page_image.save(temp_img_path, "PNG")
                         
            # Run Apple Vision OCR
            try:
                annotations = ocrmac.OCR(
                    temp_img_path, 
                    language_preference=['ar-SA'], 
                    recognition_level='accurate'
                ).recognize()
                                 
                # Extract text
                text_blocks = []
                for item in annotations:
                    if isinstance(item, (list, tuple)) and len(item) > 0:
                        text_blocks.append(str(item[0]))
                    else:
                        text_blocks.append(str(item))
                page_text = "\n".join(text_blocks)
                                 
                # Write immediately to file
                f.write(f"--- Page {i} ---\n")
                f.write(page_text)
                f.write("\n\n")
                             
            except Exception as e:
                print(f"   OCR Error on page {i}: {e}")
                             
            finally:
                # Clean up memory and file immediately
                del page_image
                del single_page_list
                if os.path.exists(temp_img_path):
                    os.remove(temp_img_path)

    print(f"-> Finished OCR! Saved to {output_txt_path}")

    # Move the PDF file to its respective processed subfolder
    try:
        shutil.move(pdf_path, destination_pdf_path)
        print(f"-> Moved PDF to {destination_pdf_path}")
    except Exception as e:
        print(f"-> Error moving {file_name}: {e}")

print("\n" + "="*40 + "\nAll folders and subfolders processed successfully!")


Found 103 PDF(s) across all folders to process.

[1/103] Processing: 2013/2_د_3_2012.pdf
-> Total pages: 6
   Processing page 1/6...
   Processing page 2/6...
   Processing page 3/6...
   Processing page 4/6...
   Processing page 5/6...
   Processing page 6/6...
-> Finished OCR! Saved to ./ocr_results/2013/2_د_3_2012.txt
-> Moved PDF to ./processed_pdfs/2013/2_د_3_2012.pdf

[2/103] Processing: 2013/4_م.ت_2_13.pdf
-> Total pages: 7
   Processing page 1/7...
   Processing page 2/7...
   Processing page 3/7...
   Processing page 4/7...
   Processing page 5/7...
   Processing page 6/7...
   Processing page 7/7...
-> Finished OCR! Saved to ./ocr_results/2013/4_م.ت_2_13.txt
-> Moved PDF to ./processed_pdfs/2013/4_م.ت_2_13.pdf

[3/103] Processing: 2013/12_ح_1_2013.pdf
-> Total pages: 6
   Processing page 1/6...
   Processing page 2/6...
   Processing page 3/6...
   Processing page 4/6...
   Processing page 5/6...
   Processing page 6/6...
-> Finished OCR! Saved to ./ocr_results/2013/12_ح_1_20

In [1]:
import os
import shutil
from pdf2image import pdfinfo_from_path, convert_from_path
from ocrmac import ocrmac

# Define main folder paths
INPUT_FOLDER = "./هيئة التشريع و الرأي القانوني"  # Root folder containing arbitrary subfolders
DONE_ROOT = "./processed_pdfs"     # Moved PDFs will mirror subfolders here

# Collect all PDF files across all subfolders
pdf_tasks = []
for root, dirs, files in os.walk(INPUT_FOLDER):
    for file in files:
        if file.lower().endswith(".pdf"):
            full_path = os.path.join(root, file)
            # Calculate the relative path from the input folder
            rel_path = os.path.relpath(full_path, INPUT_FOLDER)
            pdf_tasks.append((full_path, rel_path))

if not pdf_tasks:
    print(f"No PDF files found inside '{INPUT_FOLDER}' or its subfolders.")
    exit()

print(f"Found {len(pdf_tasks)} PDF(s) across all folders to process.\n" + "="*40)

# Loop through each PDF file task
for file_index, (pdf_path, rel_path) in enumerate(pdf_tasks, 1):
    file_name = os.path.basename(pdf_path)
    
    # 1. Output TXT goes directly into the PDF's current directory
    current_output_dir = os.path.dirname(pdf_path)
    
    # 2. Destination directory for moving the processed PDF
    rel_dir = os.path.dirname(rel_path)
    current_done_dir = os.path.join(DONE_ROOT, rel_dir)
    os.makedirs(current_done_dir, exist_ok=True)
    
    # Define file target paths
    txt_name = os.path.splitext(file_name)[0] + ".txt"
    output_txt_path = os.path.join(current_output_dir, txt_name)
    destination_pdf_path = os.path.join(current_done_dir, file_name)
         
    print(f"\n[{file_index}/{len(pdf_tasks)}] Processing: {rel_path}")
         
    try:
        # Get total page count
        info = pdfinfo_from_path(pdf_path)
        total_pages = info["Pages"]
        print(f"-> Total pages: {total_pages}")
    except Exception as e:
        print(f"-> Error reading PDF info for {file_name}: {e}. Skipping file.")
        continue

    # Open the output text file inside the same folder as the PDF
    with open(output_txt_path, "w", encoding="utf-8") as f:
        for i in range(1, total_pages + 1):
            print(f"   Processing page {i}/{total_pages}...")
                         
            try:
                single_page_list = convert_from_path(
                    pdf_path, 
                    dpi=300, 
                    first_page=i, 
                    last_page=i
                )
            except Exception as e:
                print(f"   Error converting page {i}: {e}")
                continue
                             
            if not single_page_list:
                continue
                             
            page_image = single_page_list[0]
            temp_img_path = f"temp_page_{file_index}_{i}.png"
                         
            # Save temporary page image to disk
            page_image.save(temp_img_path, "PNG")
                         
            # Run Apple Vision OCR
            try:
                annotations = ocrmac.OCR(
                    temp_img_path, 
                    language_preference=['ar-SA'], 
                    recognition_level='accurate'
                ).recognize()
                                 
                # Extract text lines
                text_blocks = []
                for item in annotations:
                    if isinstance(item, (list, tuple)) and len(item) > 0:
                        text_blocks.append(str(item[0]))
                    else:
                        text_blocks.append(str(item))
                page_text = "\n".join(text_blocks)
                                 
                # Write page content to output .txt file
                f.write(f"--- Page {i} ---\n")
                f.write(page_text)
                f.write("\n\n")
                             
            except Exception as e:
                print(f"   OCR Error on page {i}: {e}")
                             
            finally:
                # Cleanup temporary image file and memory
                del page_image
                del single_page_list
                if os.path.exists(temp_img_path):
                    os.remove(temp_img_path)

    print(f"-> Finished OCR! Saved text file to: {output_txt_path}")

    # Move processed PDF out to its corresponding directory in processed_pdfs
    try:
        shutil.move(pdf_path, destination_pdf_path)
        print(f"-> Moved PDF to: {destination_pdf_path}")
    except Exception as e:
        print(f"-> Error moving {file_name}: {e}")

print("\n" + "="*40 + "\nAll folders and subfolders processed successfully!")

Found 701 PDF(s) across all folders to process.

[1/701] Processing: المعاهدات/الجمارك/قانون رقم (2).pdf
-> Total pages: 54
   Processing page 1/54...
   Processing page 2/54...
   Processing page 3/54...
   Processing page 4/54...
   Processing page 5/54...
   Processing page 6/54...
   Processing page 7/54...
   Processing page 8/54...
   Processing page 9/54...
   Processing page 10/54...
   Processing page 11/54...
   Processing page 12/54...
   Processing page 13/54...
   Processing page 14/54...
   Processing page 15/54...
   Processing page 16/54...
   Processing page 17/54...
   Processing page 18/54...
   Processing page 19/54...
   Processing page 20/54...
   Processing page 21/54...
   Processing page 22/54...
   Processing page 23/54...
   Processing page 24/54...
   Processing page 25/54...
   Processing page 26/54...
   Processing page 27/54...
   Processing page 28/54...
   Processing page 29/54...
   Processing page 30/54...
   Processing page 31/54...
   Processing pag

/Users/zainababdulwahab/GA_DSB/ga_dsb_env/lib/python3.12/site-packages/PIL/Image.py:3368: DecompressionBombWarning: Image size (136003200 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


   Processing page 2/530...


/Users/zainababdulwahab/GA_DSB/ga_dsb_env/lib/python3.12/site-packages/PIL/Image.py:3368: DecompressionBombWarning: Image size (136003200 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


   Processing page 3/530...


...

   Processing page 12/530...


/Users/zainababdulwahab/GA_DSB/ga_dsb_env/lib/python3.12/site-packages/PIL/Image.py:3368: DecompressionBombWarning: Image size (136003200 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


   Processing page 13/530...


/Users/zainababdulwahab/GA_DSB/ga_dsb_env/lib/python3.12/site-packages/PIL/Image.py:3368: DecompressionBombWarning: Image size (136003200 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
import os
import shutil
from PIL import Image
from pdf2image import pdfinfo_from_path, convert_from_path
from ocrmac import ocrmac

# Disable PIL decompression bomb protection for high-resolution PDF pages
Image.MAX_IMAGE_PIXELS = None

# Define main folder paths
INPUT_FOLDER = "./هيئة التشريع و الرأي القانوني"  # Root folder containing arbitrary subfolders
DONE_ROOT = "./processed_pdfs"     # Moved PDFs will mirror subfolders here

# Collect all PDF files across all subfolders
pdf_tasks = []
for root, dirs, files in os.walk(INPUT_FOLDER):
    for file in files:
        if file.lower().endswith(".pdf"):
            full_path = os.path.join(root, file)
            # Calculate the relative path from the input folder
            rel_path = os.path.relpath(full_path, INPUT_FOLDER)
            pdf_tasks.append((full_path, rel_path))

if not pdf_tasks:
    print(f"No PDF files found inside '{INPUT_FOLDER}' or its subfolders.")
    exit()

print(f"Found {len(pdf_tasks)} PDF(s) across all folders to process.\n" + "="*40)

# Loop through each PDF file task
for file_index, (pdf_path, rel_path) in enumerate(pdf_tasks, 1):
    file_name = os.path.basename(pdf_path)
    
    # 1. Output TXT goes directly into the PDF's current directory
    current_output_dir = os.path.dirname(pdf_path)
    
    # 2. Destination directory for moving the processed PDF
    rel_dir = os.path.dirname(rel_path)
    current_done_dir = os.path.join(DONE_ROOT, rel_dir)
    os.makedirs(current_done_dir, exist_ok=True)
    
    # Define file target paths
    txt_name = os.path.splitext(file_name)[0] + ".txt"
    output_txt_path = os.path.join(current_output_dir, txt_name)
    destination_pdf_path = os.path.join(current_done_dir, file_name)
         
    print(f"\n[{file_index}/{len(pdf_tasks)}] Processing: {rel_path}")
         
    try:
        # Get total page count
        info = pdfinfo_from_path(pdf_path)
        total_pages = info["Pages"]
        print(f"-> Total pages: {total_pages}")
    except Exception as e:
        print(f"-> Error reading PDF info for {file_name}: {e}. Skipping file.")
        continue

    # Open the output text file inside the same folder as the PDF
    with open(output_txt_path, "w", encoding="utf-8") as f:
        for i in range(1, total_pages + 1):
            print(f"   Processing page {i}/{total_pages}...")
                         
            try:
                single_page_list = convert_from_path(
                    pdf_path, 
                    dpi=200,  # 200 DPI optimizes speed and prevents excessive memory usage
                    first_page=i, 
                    last_page=i
                )
            except Exception as e:
                print(f"   Error converting page {i}: {e}")
                continue
                             
            if not single_page_list:
                continue
                             
            page_image = single_page_list[0]
            temp_img_path = f"temp_page_{file_index}_{i}.png"
                         
            # Save temporary page image to disk
            page_image.save(temp_img_path, "PNG")
                         
            # Run Apple Vision OCR
            try:
                annotations = ocrmac.OCR(
                    temp_img_path, 
                    language_preference=['ar-SA'], 
                    recognition_level='accurate'
                ).recognize()
                                 
                # Extract text lines
                text_blocks = []
                for item in annotations:
                    if isinstance(item, (list, tuple)) and len(item) > 0:
                        text_blocks.append(str(item[0]))
                    else:
                        text_blocks.append(str(item))
                page_text = "\n".join(text_blocks)
                                 
                # Write page content to output .txt file
                f.write(f"--- Page {i} ---\n")
                f.write(page_text)
                f.write("\n\n")
                             
            except Exception as e:
                print(f"   OCR Error on page {i}: {e}")
                             
            finally:
                # Cleanup temporary image file and memory
                del page_image
                del single_page_list
                if os.path.exists(temp_img_path):
                    os.remove(temp_img_path)

    print(f"-> Finished OCR! Saved text file to: {output_txt_path}")

    # Move processed PDF out to its corresponding directory in processed_pdfs
    try:
        shutil.move(pdf_path, destination_pdf_path)
        print(f"-> Moved PDF to: {destination_pdf_path}")
    except Exception as e:
        print(f"-> Error moving {file_name}: {e}")

print("\n" + "="*40 + "\nAll folders and subfolders processed successfully!")

Found 270 PDF(s) across all folders to process.

[1/270] Processing: المعاهدات/البيئة/قانون رقم (32) لسنة 2005.pdf
-> Total pages: 530
   Processing page 1/530...
   Processing page 2/530...
   Processing page 3/530...
   Processing page 4/530...
   Processing page 5/530...
   Processing page 6/530...
   Processing page 7/530...
   Processing page 8/530...
   Processing page 9/530...
   Processing page 10/530...
   Processing page 11/530...
   Processing page 12/530...
   Processing page 13/530...
   Processing page 14/530...
   Processing page 15/530...
   Processing page 16/530...
   Processing page 17/530...
   Processing page 18/530...
   Processing page 19/530...
   Processing page 20/530...
   Processing page 21/530...
   Processing page 22/530...
   Processing page 23/530...
   Processing page 24/530...
   Processing page 25/530...
   Processing page 26/530...
   Processing page 27/530...
   Processing page 28/530...
   Processing page 29/530...
   Processing page 30/530...
   P